# Unified Model Loader Demo - OpenEye Multi-Framework Support

This notebook demonstrates the **unified model loader** feature of OpenEye, which allows you to load models from different frameworks (TensorFlow, PyTorch, ONNX) using a single, consistent API.

## Features Demonstrated
- Auto-detection of model formats
- Loading models from multiple frameworks
- Layer adaptation for OpenEye hardware
- Quantization support for all frameworks
- Hardware mapping workflow

## Setup and Imports

In [ ]:
import sys
import os

# Add OpenEye source to path
openeye_base = os.path.abspath(os.path.join(os.pardir, os.pardir))
sys.path.insert(0, os.path.join(openeye_base, "src"))

print(f"OpenEye base directory: {openeye_base}")

In [ ]:
# Import unified model loader
from open_eye.model_loader import load_model, detect_model_format
from open_eye.layer_adapter import adapt_layers_for_keras

import numpy as np
print("✅ OpenEye unified model loader imported successfully")

## Part 1: Format Detection

The unified loader can automatically detect the format of your model file.

In [ ]:
# Test format detection
test_files = [
    'mnist_quantized_model.tflite',
    'mnist_quantized_model.onnx',
    'mnist_model_quantized.pth',
    'mnist_cnn_model.h5'
]

print("Format Detection Results:")
print("=" * 50)
for filepath in test_files:
    if os.path.exists(filepath):
        detected_format = detect_model_format(filepath)
        print(f"File: {filepath:35s} → Format: {detected_format}")
    else:
        print(f"File: {filepath:35s} → Not found (skipped)")

## Part 2: Loading Models from Different Frameworks

### 2.1 Load TensorFlow Lite Model

In [ ]:
# Load TFLite model (auto-detection)
if os.path.exists('mnist_quantized_model.tflite'):
    print("Loading TensorFlow Lite model...")
    tflite_model = load_model('mnist_quantized_model.tflite')
    print(f"✅ TFLite model loaded: {len(tflite_model.layers)} layers")
    print(f"   Input shape: {tflite_model.input_shape}")
    print(f"   Quantized: {tflite_model.is_quantized}")
    
    # Show layer structure
    print("\n   Layer structure:")
    for i, layer in enumerate(tflite_model.layers):
        print(f"     Layer {i}: {layer.__class__.__name__}")
else:
    print("⚠️  TFLite model not found. Please run mnist.ipynb first.")

### 2.2 Load ONNX Model

In [ ]:
# Load ONNX model (auto-detection)
if os.path.exists('mnist_quantized_model.onnx'):
    print("Loading ONNX model...")
    onnx_model = load_model('mnist_quantized_model.onnx')
    print(f"✅ ONNX model loaded: {len(onnx_model.layers)} layers")
    print(f"   Input shape: {onnx_model.input_shape}")
    print(f"   Quantized: {onnx_model.is_quantized}")
    
    # Show layer structure
    print("\n   Layer structure:")
    for i, layer in enumerate(onnx_model.layers):
        print(f"     Layer {i}: {layer.__class__.__name__}")
else:
    print("⚠️  ONNX model not found. Please run mnist_onnx.ipynb first.")

### 2.3 Load PyTorch Model

In [ ]:
# Load PyTorch model (requires explicit input shape)
if os.path.exists('mnist_model_quantized.pth'):
    print("Loading PyTorch model...")
    pytorch_model = load_model(
        'mnist_model_quantized.pth',
        format='pytorch',
        input_shape=(1, 1, 28, 28)  # PyTorch needs input shape
    )
    print(f"✅ PyTorch model loaded: {len(pytorch_model.layers)} layers")
    print(f"   Input shape: {pytorch_model.input_shape}")
    print(f"   Quantized: {pytorch_model.is_quantized}")
    
    # Show layer structure
    print("\n   Layer structure:")
    for i, layer in enumerate(pytorch_model.layers):
        print(f"     Layer {i}: {layer.__class__.__name__}")
else:
    print("⚠️  PyTorch model not found. Please run mnist_pytorch.ipynb first.")

### 2.4 Load Keras Model

In [ ]:
# Load Keras model
if os.path.exists('mnist_cnn_model.h5'):
    print("Loading Keras model...")
    keras_model = load_model('mnist_cnn_model.h5', format='keras')
    print(f"✅ Keras model loaded: {len(keras_model.layers)} layers")
    print(f"   Input shape: {keras_model.input_shape}")
    
    # Show layer structure
    print("\n   Layer structure:")
    for i, layer in enumerate(keras_model.layers):
        print(f"     Layer {i}: {layer.__class__.__name__}")
else:
    print("⚠️  Keras model not found. Please run mnist.ipynb first.")

## Part 3: Layer Adaptation for OpenEye Hardware

All models need to be adapted to OpenEye's internal layer representation before hardware mapping.

In [ ]:
# Adapt layers for hardware (using TFLite model as example)
if 'tflite_model' in locals():
    print("Adapting layers for OpenEye hardware...")
    adapted_layers = adapt_layers_for_keras(tflite_model.layers)
    
    print(f"\n✅ Adapted {len(adapted_layers)} layers for OpenEye")
    print("\nAdapted layer types:")
    for i, layer in enumerate(adapted_layers):
        layer_name = layer.name if hasattr(layer, 'name') else 'Unknown'
        output_shape = layer.output.shape if hasattr(layer, 'output') else 'N/A'
        print(f"  {i}: {layer_name:15s} → Shape: {output_shape}")

## Part 4: Hardware Mapping Workflow

Once layers are adapted, they can be mapped to OpenEye hardware using LayerParameters.

In [ ]:
# Import hardware mapping tools
from open_eye.layer_parameters import LayerParameters
from open_eye.pe_cluster_test_utils import OpenEyeParameters

# Define hardware parameters
hw_params = OpenEyeParameters(
    Clusters_X=2,
    Clusters_Y=2,
    PEs_X=2,
    PEs_Y=3,
    IACT_Bitwidth=8,
    WGHT_Bitwidth=8,
    PSUM_Bitwidth=20
)

print("OpenEye Hardware Configuration:")
print("=" * 50)
print(f"  Clusters: {hw_params.Clusters_X} × {hw_params.Clusters_Y} = {hw_params.Clusters_X * hw_params.Clusters_Y}")
print(f"  PEs per cluster: {hw_params.PEs_X} × {hw_params.PEs_Y} = {hw_params.PEs_X * hw_params.PEs_Y}")
print(f"  Total PEs: {hw_params.Clusters_X * hw_params.Clusters_Y * hw_params.PEs_X * hw_params.PEs_Y}")
print(f"  Data widths: IACT={hw_params.IACT_Bitwidth}b, WGHT={hw_params.WGHT_Bitwidth}b, PSUM={hw_params.PSUM_Bitwidth}b")

In [ ]:
# Map first convolutional layer to hardware (example)
if 'adapted_layers' in locals() and len(adapted_layers) > 0:
    print("Mapping first layer to hardware...")
    
    # Find first Conv2D layer
    first_conv = None
    for layer in adapted_layers:
        layer_name = layer.name if hasattr(layer, 'name') else ''
        if 'conv' in layer_name.lower():
            first_conv = layer
            break
    
    if first_conv:
        print(f"\nMapping Conv2D layer:")
        if hasattr(first_conv, 'filters'):
            print(f"  Filters: {first_conv.filters}")
        if hasattr(first_conv, 'kernel_size'):
            print(f"  Kernel: {first_conv.kernel_size}")
        if hasattr(first_conv, 'input'):
            print(f"  Input: {first_conv.input.shape}")
        if hasattr(first_conv, 'output'):
            print(f"  Output: {first_conv.output.shape}")
        
        # Create LayerParameters
        try:
            layer_params = LayerParameters(
                layer_dict=first_conv,
                hw_params=hw_params
            )
            
            print(f"\n✅ Layer mapped to hardware successfully")
            if hasattr(layer_params, 'pe_utilization'):
                print(f"  PE utilization: {layer_params.pe_utilization:.1f}%")
            if hasattr(layer_params, 'memory_required'):
                print(f"  Memory required: {layer_params.memory_required} bytes")
        except Exception as e:
            print(f"⚠️  Mapping failed: {e}")
    else:
        print("⚠️  No Conv2D layer found in adapted layers")

## Part 5: Cross-Framework Comparison

Compare the same model loaded from different frameworks.

In [ ]:
# Compare models from different frameworks
print("Cross-Framework Model Comparison")
print("=" * 70)

models = []
if 'tflite_model' in locals():
    models.append(('TFLite', tflite_model))
if 'onnx_model' in locals():
    models.append(('ONNX', onnx_model))
if 'pytorch_model' in locals():
    models.append(('PyTorch', pytorch_model))
if 'keras_model' in locals():
    models.append(('Keras', keras_model))

if models:
    print(f"{'Framework':<12} {'Layers':<8} {'Input Shape':<20} {'Quantized':<10}")
    print("-" * 70)
    for name, model in models:
        quantized = model.is_quantized if hasattr(model, 'is_quantized') else 'N/A'
        print(f"{name:<12} {len(model.layers):<8} {str(model.input_shape):<20} {str(quantized):<10}")
    
    print("\n✅ All frameworks produce compatible OpenEye models!")
else:
    print("⚠️  No models loaded. Please run the framework-specific notebooks first.")

## Part 6: Best Practices

### Recommendations for Multi-Framework Usage

1. **Always quantize** models before deployment (4× smaller, faster)
2. **Use auto-detection** when possible (simpler code)
3. **Provide input_shape** for PyTorch models (required)
4. **Adapt layers** before hardware mapping (ensures compatibility)
5. **Keep kernel sizes** ≤ 5×5 for optimal hardware usage

### Supported Layer Types

- Conv2D (all frameworks)
- MaxPool2D (all frameworks)
- Dense/Linear (all frameworks)
- ReLU (all frameworks)
- Flatten (all frameworks)

## Summary

This notebook demonstrated:

✅ Automatic format detection for all supported frameworks  
✅ Unified API for loading TFLite, ONNX, PyTorch, and Keras models  
✅ Layer adaptation for OpenEye hardware compatibility  
✅ Hardware mapping workflow  
✅ Cross-framework model comparison  

**Next Steps:**
- Run individual framework notebooks (mnist.ipynb, mnist_pytorch.ipynb, mnist_onnx.ipynb)
- Experiment with your own models
- Deploy on OpenEye hardware simulator

For more information, see the [Multi-Framework Support Documentation](../source/tutorial/multi_framework_support.rst)